## Air Quality Index (AQI) — Interactive 3D Map with pydeck

This notebook queries AQI data for the Pyrenees corridor — the French regions of Occitanie and Nouvelle-Aquitaine, and the Spanish regions of Aragon, Catalonia and Navarre — from Neo4j and renders an interactive 3D map using pydeck (deck.gl). Cities are shown as 3D columns rising from the map — height and color both encode AQI value. The map is fully interactive: pan, zoom, tilt, and rotate to explore the data.

### Before running the notebook

Export the following in your shell:

```bash
export NEO4J_URI=bolt://127.0.0.1:7687
export NEO4J_USERNAME=your_username_here
export NEO4J_PASSWORD=your_password_here
export NEO4J_DATABASE=your_database_name_here
```

### Install and Import Packages

In [1]:
%pip install neo4j==5.28.4 \
             pydeck==0.9.1 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


In [2]:
import os
import pydeck as pdk

from neo4j import GraphDatabase

### Connection Settings

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

print("Credentials set.")

Credentials set.


### Step 1: Query Latest AQI per City from Neo4j

We fetch the most recent Reading for each City node, along with all fields needed for the tooltip.

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


In [5]:
query = """
    MATCH (c:City)-[:HAS_READING]->(r:Reading)
    WITH c, r ORDER BY r.timestamp DESC
    WITH c, collect(r)[0]        AS latest
    RETURN c.name                AS city,
           c.country             AS country,
           c.lat                 AS lat,
           c.lon                 AS lon,
           latest.aqi_us         AS aqi_us,
           latest.aqi_category   AS aqi_category,
           latest.main_pollutant AS main_pollutant,
           latest.temperature    AS temperature,
           latest.humidity       AS humidity,
           latest.wind_speed     AS wind_speed,
           latest.timestamp      AS timestamp
    ORDER BY latest.aqi_us DESC
"""

with driver.session(database = NEO4J_DATABASE) as session:
    result = session.run(query)
    records = [dict(r) for r in result]

driver.close()

print(f"Cities loaded: {len(records)}")
print(f"AQI range: {min(r['aqi_us'] for r in records)} - {max(r['aqi_us'] for r in records)}")

Cities loaded: 102
AQI range: 1 - 157


### Step 2: Prepare Data for pydeck

We add color (standard US AQI scale as RGB), column height scaled from AQI, and a formatted tooltip string for each city.

In [6]:
# Standard US AQI color scale as RGB tuples
def aqi_color(aqi):
    if aqi <= 50:  return [  0, 228,   0]  # Good
    if aqi <= 100: return [255, 255,   0]  # Moderate
    if aqi <= 150: return [255, 126,   0]  # Unhealthy for Sensitive Groups
    if aqi <= 200: return [255,   0,   0]  # Unhealthy
    if aqi <= 300: return [143,  63, 151]  # Very Unhealthy
    return                [126,   0,  35]  # Hazardous

# Column height: scale AQI to meters (visible at regional zoom level)
# 1 AQI unit = 500m, deck.gl handles large values fine
AQI_HEIGHT_SCALE = 500

for r in records:
    r["color"]  = aqi_color(r["aqi_us"])
    r["height"] = r["aqi_us"] * AQI_HEIGHT_SCALE
    r["tooltip"] = (
        f"{r['city']}, {r['country']}\n"
        f"AQI (US): {r['aqi_us']} — {r['aqi_category']}\n"
        f"Main pollutant: {r['main_pollutant']}\n"
        f"Temperature: {r['temperature']}°C   Humidity: {r['humidity']}%\n"
        f"Wind: {r['wind_speed']} m/s\n"
        f"Recorded: {str(r['timestamp'])[:19]} UTC"
    )

print(f"Data prepared: {len(records)} cities")

Data prepared: 102 cities


### Step 3: Build the pydeck Map

We use deck.gl's `ColumnLayer` for the 3D AQI spikes and `ScatterplotLayer` for ground-level city dots. The initial view is centered on the Pyrenees corridor at a 50-degree tilt to show the 3D columns clearly.

In [7]:
# Initial camera: centered on the Pyrenees corridor, tilted to show 3D columns
initial_view = pdk.ViewState(
    latitude  = 43.0,
    longitude = 1.5,
    zoom      = 5.5,
    pitch     = 50,
    bearing   = 0
)

# 3D AQI columns — height and color both encode AQI
column_layer = pdk.Layer(
    "ColumnLayer",
    data                  = records,
    get_position          = ["lon", "lat"],
    get_elevation         = "height",
    elevation_scale       = 1,
    radius                = 2000,        # much narrower — spike-like
    get_fill_color        = "color",
    get_line_color        = [255, 255, 255],
    line_width_min_pixels = 1,
    pickable              = True,
    auto_highlight        = True,
    disk_resolution       = 4            # square cross-section instead of circle
)

# Ground dots so cities are visible when zoomed out and columns overlap
scatter_layer = pdk.Layer(
    "ScatterplotLayer",
    data           = records,
    get_position   = ["lon", "lat"],
    get_radius     = 5000,
    get_fill_color = "color",
    opacity        = 0.6,
    pickable       = True
)

# Tooltip shown on hover
tooltip = {
    "text": "{tooltip}",
    "style": {
        "backgroundColor": "#1a1a2e",
        "color":           "white",
        "fontSize":        "13px",
        "padding":         "8px",
        "borderRadius":    "4px",
        "whiteSpace":      "pre-line"
    }
}

deck = pdk.Deck(
    layers             = [column_layer, scatter_layer],
    initial_view_state = initial_view,
    map_style          = "light",
    tooltip            = tooltip
)

print("Map built.")

Map built.


### Step 4: Save and Display

We save the map as a self-contained HTML file and display it inline. Use left-click to rotate, right-click to pan, scroll to zoom, and hover over any column for the full city details.

In [8]:
map_file = "aqi_pyrenees_pydeck.html"
deck.to_html(map_file, open_browser=False)
print(f"Map saved: {map_file}")

Map saved: aqi_pyrenees_pydeck.html


In [9]:
from IPython.display import IFrame
IFrame(map_file, width="100%", height=600)